# 📊 Analyse Exploratoire des Données (EDA)
## Système de Prédiction de Défaut de Crédit

**Objectif** : Comprendre la distribution des données, le taux de défaut, les corrélations et les features les plus importantes avant l'entraînement d'un modèle.


In [ ]:
import sys
sys.path.append('..')   # allow imports from project root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

from src.data.loader import load_raw_data
from src.data.cleaner import clean_and_save
from src.config import FEATURES_FILE, FIGURES_DIR

print('Libraries loaded ✅')

## 1. Chargement & Nettoyage

In [ ]:
raw = load_raw_data()
data = clean_and_save(raw)

clients  = data['clients']
credits  = data['credits']
remb     = data['remboursements']
tx       = data['transactions']
rel      = data['relations']

## 2. Vue d'ensemble des tables

In [ ]:
for name, df in data.items():
    print(f'\n=== {name} ({df.shape[0]:,} rows × {df.shape[1]} cols) ===')
    display(df.describe(include='all').T.head(20))

## 3. Distribution de la variable cible (`en_defaut`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Counts
counts = credits['en_defaut'].value_counts()
axes[0].bar(['Non-défaut (0)', 'Défaut (1)'], counts.values, color=['#2ecc71', '#e74c3c'], edgecolor='black')
for i, v in enumerate(counts.values): axes[0].text(i, v + 20, f'{v:,}', ha='center', fontsize=11)
axes[0].set_title('Nombre de crédits par classe')
axes[0].set_ylabel('Nombre de crédits')

# Pie
axes[1].pie(counts.values, labels=['Non-défaut', 'Défaut'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title(f'Taux de défaut : {counts[1]/counts.sum()*100:.1f}%')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'target_distribution.png', dpi=120)
plt.show()

## 4. Distribution des features numériques clients

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(clients['age'], bins=30, color='#3498db', edgecolor='white')
axes[0].set_title('Distribution de l\'âge'); axes[0].set_xlabel('Âge')
axes[1].hist(clients['revenu_mensuel'], bins=50, color='#9b59b6', edgecolor='white')
axes[1].set_title('Distribution du revenu mensuel'); axes[1].set_xlabel('Revenu (TND)')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'client_distributions.png', dpi=120); plt.show()

## 5. Taux de défaut par catégorie

In [ ]:
# Merge credits with clients
df_merged = credits.merge(clients[['client_id','profession','statut_kyc','ville']], on='client_id')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ['statut_kyc', 'profession', 'cycle']):
    df_rate = df_merged.groupby(str(col))['en_defaut'].mean().sort_values(ascending=False)
    df_rate.plot(kind='bar', ax=ax, color='#e67e22', edgecolor='black')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.set_title(f'Taux de défaut par {col}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'default_rate_by_category.png', dpi=120)
plt.show()

## 6. DTI vs Défaut

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Box plot
credits.boxplot(column='dti', by='en_defaut', ax=axes[0])
axes[0].set_title('DTI par statut de défaut')
axes[0].set_xlabel('en_defaut (0=Non, 1=Oui)')
plt.sca(axes[0]); plt.title('DTI par statut de défaut')

# Violin
sns.violinplot(data=credits, x='en_defaut', y='dti', palette=['#2ecc71','#e74c3c'], ax=axes[1])
axes[1].set_title('Distribution DTI par classe')
axes[1].set_xlabel('en_defaut')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dti_vs_defaut.png', dpi=120)
plt.show()

## 7. Retards de remboursement

In [ ]:
# Aggregate retard by credit
remb_agg = remb.groupby('credit_id')['retard_jours'].agg(['mean', 'max']).reset_index()
remb_agg = remb_agg.merge(credits[['credit_id', 'en_defaut']], on='credit_id')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes, ['mean', 'max'], ['Retard moyen (jours)', 'Retard max (jours)']):
    sns.histplot(data=remb_agg, x=col, hue='en_defaut', bins=40, ax=ax,
                 palette={0:'#2ecc71', 1:'#e74c3c'}, alpha=0.7, element='step')
    ax.set_title(label); ax.set_xlabel(label)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'remboursement_delays.png', dpi=120)
plt.show()

## 8. Matrice de corrélation (feature matrix)

In [ ]:
if FEATURES_FILE.exists():
    df_feat = pd.read_parquet(FEATURES_FILE)
    print(f'Feature matrix: {df_feat.shape}')

    num_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
    corr = df_feat[num_cols].corr()

    fig, ax = plt.subplots(figsize=(14, 11))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
                linewidths=0.3, ax=ax, vmin=-1, vmax=1)
    ax.set_title('Matrice de corrélation – Feature Matrix')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'correlation_matrix.png', dpi=120)
    plt.show()

    # Correlation with target
    target_corr = corr['en_defaut'].drop('en_defaut').sort_values(key=abs, ascending=False)
    print('\nTop correlations with en_defaut:')
    print(target_corr.head(15).to_string())
else:
    print('Run `python -m src.features.engineering` first to generate features.parquet')

## 9. Transactions suspectes par statut de défaut

In [ ]:
tx_agg = tx.groupby('client_id').agg(
    n_suspect=('suspect', 'sum'),
    n_transac=('transaction_id', 'count')
).reset_index()

# Get default status from credits (any credit in default → client is high-risk)
client_default = credits.groupby('client_id')['en_defaut'].max().reset_index()
tx_agg = tx_agg.merge(client_default, on='client_id', how='left')

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=tx_agg, x='en_defaut', y='n_suspect',
            palette={0:'#2ecc71', 1:'#e74c3c'}, ax=ax)
ax.set_title('Transactions suspectes par statut de défaut')
ax.set_xlabel('Défaut (0=Non, 1=Oui)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'suspect_transactions.png', dpi=120)
plt.show()

---
## ✅ Conclusions
- **Taux de défaut** : ~11.5% → déséquilibre de classes à traiter
- **DTI** : forte corrélation avec le défaut (DTI élevé → risque élevé)
- **Retard de remboursement** : très discriminant (max_retard sépare bien les classes)
- **Statut KYC RISQUE** : taux de défaut nettement supérieur
- **Revenu mensuel** : corrélation négative avec le défaut (plus le revenu est élevé, moins le risque est élevé)

**Prochaine étape** → `python -m src.features.engineering` puis `python -m src.models.train`